# LeetCode 819 — Most Common Word bằng Python

**Link đề:** https://leetcode.com/problems/most-common-word/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, String, Counting

```python
class Solution:
    def mostCommonWord(self, paragraph: str, banned: list[str]) -> str:
        ...
```

## Mục tiêu học
1. **Tách từ** (tokenize) từ một đoạn văn có dấu câu: tự viết bằng vòng lặp ký tự, bằng `replace` + `split`, bằng `re.findall`.
2. Chuẩn hóa chữ thường bằng `.lower()`, kiểm tra chữ cái bằng `.isalpha()`.
3. **Đếm tần suất** bằng `dict` và `collections.Counter`; loại từ cấm bằng `set`.
4. Tìm phần tử có tần suất lớn nhất: `max(d, key=d.get)`, `Counter.most_common()`.
5. Trace, chứng minh, Time/Space, các lỗi "ăn điểm" khi tách từ, luyện biến thể (top-k từ).

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Cho chuỗi `paragraph` và mảng các từ bị cấm `banned`. Trả về **từ xuất hiện nhiều nhất không bị cấm**. Đề bảo đảm có ít nhất một từ không bị cấm và **đáp án là duy nhất**.

- Từ trong `paragraph` **không phân biệt hoa/thường**; đáp án trả về ở dạng **chữ thường**.
- Từ **không chứa dấu câu** — dấu câu đứng sát từ (như `"ball,"`) phải bỏ đi.

**Ví dụ 1:** `paragraph = "Bob hit a ball, the hit BALL flew far after it was hit."`, `banned = ["hit"]` → `"ball"`
- `"hit"` xuất hiện 3 lần nhưng bị cấm.
- `"ball"` xuất hiện 2 lần (`"ball,"` và `"BALL"`), nhiều nhất trong các từ còn lại.

**Ví dụ 2:** `paragraph = "a."`, `banned = []` → `"a"`

**Constraints:**
- `1 <= paragraph.length <= 1000`
- `paragraph` gồm chữ cái tiếng Anh, dấu cách `' '`, hoặc một trong các ký hiệu `!?',;.`
- `0 <= banned.length <= 100`; `1 <= banned[i].length <= 10`; `banned[i]` chỉ chữ thường.

In [1]:
p1 = "Bob hit a ball, the hit BALL flew far after it was hit."
banned1 = ["hit"]
p2 = "a."
banned2 = []
print("split() thô:", p1.split())

split() thô: ['Bob', 'hit', 'a', 'ball,', 'the', 'hit', 'BALL', 'flew', 'far', 'after', 'it', 'was', 'hit.']


Nhìn kết quả `split()` thô: `'ball,'`, `'BALL'`, `'hit.'` — dấu câu dính vào từ và hoa/thường lẫn lộn. **Phần khó của bài là tách từ cho đúng**, còn đếm thì đơn giản.

## 2. Nền tảng Python: các phương thức chuỗi cần dùng

| Phương thức | Tác dụng | Ví dụ |
|---|---|---|
| `s.lower()` | đổi thành chữ thường | `"BaLL".lower() → "ball"` |
| `ch.isalpha()` | ký tự là chữ cái? | `"a".isalpha() → True`, `",".isalpha() → False` |
| `s.split()` | tách theo **mọi khoảng trắng**, bỏ chuỗi rỗng | `"a  b ".split() → ['a', 'b']` |
| `s.replace(a, b)` | thay mọi `a` bằng `b` | `"a,b".replace(",", " ") → "a b"` |

In [2]:
print("BaLL".lower())
print("a".isalpha(), ",".isalpha(), "'".isalpha(), " ".isalpha())
print("a  b ".split(), "| với tham số:", "a  b ".split(" "))
print("a,b;c".replace(",", " ").replace(";", " "))

ball
True False False False
['a', 'b'] | với tham số: ['a', '', 'b', '']
a b c


**Chú ý quan trọng:** `split()` **không tham số** gộp nhiều dấu cách liên tiếp và bỏ chuỗi rỗng; còn `split(" ")` thì sinh chuỗi rỗng `''` giữa hai dấu cách liền nhau. Khi thay dấu câu bằng dấu cách, ta sẽ tạo ra nhiều dấu cách liền nhau → phải dùng `split()` không tham số.

## 3. Nền tảng: đếm tần suất bằng `dict` và `Counter`

Mẫu đếm bằng dict thường:
```python
count[w] = count.get(w, 0) + 1
```
`d.get(key, default)` trả `d[key]` nếu có, ngược lại trả `default` (không lỗi `KeyError`).

`Counter` làm sẵn việc này: `Counter(list_từ)`.

In [3]:
words = ["bob", "hit", "a", "ball", "hit"]
count = {}
for w in words:
    count[w] = count.get(w, 0) + 1
print(count)

from collections import Counter
c = Counter(words)
print(c, "| c['hit'] =", c["hit"], "| c['zzz'] =", c["zzz"])
print("most_common(2):", c.most_common(2))

{'bob': 1, 'hit': 2, 'a': 1, 'ball': 1}
Counter({'hit': 2, 'bob': 1, 'a': 1, 'ball': 1}) | c['hit'] = 2 | c['zzz'] = 0
most_common(2): [('hit', 2), ('bob', 1)]


**Vài thao tác `Counter` hữu ích:** `update(iterable)` cộng thêm số đếm; `c1 + c2` gộp hai bộ đếm; `most_common()` không tham số trả **mọi** cặp, sắp giảm dần theo số lần.

In [4]:
c = Counter("abca")
c.update(["a", "z"])
print(c)
print(c + Counter("zz"))
print(c.most_common())

Counter({'a': 3, 'b': 1, 'c': 1, 'z': 1})
Counter({'a': 3, 'z': 3, 'b': 1, 'c': 1})
[('a', 3), ('b', 1), ('c', 1), ('z', 1)]


### 3.1. Lấy key có value lớn nhất: `max(d, key=d.get)`

`max(d)` trên dict duyệt **các key**. Thêm `key=d.get` để so sánh theo **value** của mỗi key → trả về key có value lớn nhất.

In [5]:
d = {"bob": 1, "ball": 2, "a": 1}
print(max(d), "<- so theo thứ tự chữ cái của key")
print(max(d, key=d.get), "<- so theo value")

bob <- so theo thứ tự chữ cái của key
ball <- so theo value


## 4. Cách 1 — Tách từ thủ công bằng vòng lặp ký tự

Duyệt từng ký tự, gom các **chữ cái liên tiếp** thành một từ (dạng chữ thường). Gặp ký tự không phải chữ cái (dấu cách hoặc dấu câu) → **kết thúc** từ đang gom (nếu có). Đừng quên từ cuối cùng khi hết chuỗi.

In [6]:
def tokenize_manual(paragraph: str) -> list[str]:
    words, cur = [], []
    for ch in paragraph:
        if ch.isalpha():
            cur.append(ch.lower())       # đang ở trong một từ
        elif cur:                         # gặp ký tự phân cách và đang có từ dở
            words.append("".join(cur))
            cur = []
    if cur:                               # từ cuối (khi paragraph kết thúc bằng chữ cái)
        words.append("".join(cur))
    return words

print(tokenize_manual(p1))
print(tokenize_manual(p2), tokenize_manual("a, b,c"), tokenize_manual("Hello"))

['bob', 'hit', 'a', 'ball', 'the', 'hit', 'ball', 'flew', 'far', 'after', 'it', 'was', 'hit']
['a'] ['a', 'b', 'c'] ['hello']


In [7]:
def most_common_manual(paragraph: str, banned: list[str]) -> str:
    banned_set = set(banned)
    count = {}
    for w in tokenize_manual(paragraph):
        if w not in banned_set:
            count[w] = count.get(w, 0) + 1
    return max(count, key=count.get)

print(most_common_manual(p1, banned1), most_common_manual(p2, banned2))

ball a


**Độ phức tạp cách 1:** `P = len(paragraph)`, `B` = tổng độ dài `banned`. Tạo set cấm `O(B)`, tách từ `O(P)`, đếm `O(P)` → `O(P + B)` thời gian, `O(P + B)` bộ nhớ.

Ưu: hiểu rõ từng bước, không phụ thuộc danh sách dấu câu. Nhược: code dài.

## 5. Nút thắt và insight — chuẩn hóa rồi mới tách

Thay vì tự gom ký tự, ta **chuẩn hóa** cả đoạn văn trước:
1. `lower()` toàn bộ.
2. Thay **mọi ký tự không phải chữ cái** thành dấu cách.
3. `split()` không tham số.

Sau đó đếm bằng `Counter`, bỏ từ cấm bằng `set`. Ba dòng là xong.

In [8]:
normalized = "".join(ch.lower() if ch.isalpha() else " " for ch in p1)
print(repr(normalized))
print(normalized.split())

'bob hit a ball  the hit ball flew far after it was hit '
['bob', 'hit', 'a', 'ball', 'the', 'hit', 'ball', 'flew', 'far', 'after', 'it', 'was', 'hit']


### 5.1. Đọc kỹ dòng chuẩn hóa: generator bên trong `"".join(...)`

`"".join(X for ch in s)` nhận một **generator expression**: sinh lần lượt từng mảnh `X` (ở đây là một ký tự) rồi nối lại — không cần tạo list trung gian. Biểu thức `ch.lower() if ch.isalpha() else " "` là toán tử điều kiện một dòng: chữ cái giữ lại (dạng thường), còn lại thành dấu cách.

In [9]:
demo = "Hi, BOB!"
pieces = [ch.lower() if ch.isalpha() else " " for ch in demo]
print(pieces)
print(repr("".join(pieces)), "->", "".join(pieces).split())

['h', 'i', ' ', ' ', 'b', 'o', 'b', ' ']
'hi  bob ' -> ['hi', 'bob']


## 6. Lời giải chính — chuẩn hóa + `Counter` + `set`

```python
banned_set = set(banned)                                              # (1) tra cứu O(1)
text = "".join(c.lower() if c.isalpha() else " " for c in paragraph)  # (2) chuẩn hóa
count = Counter(w for w in text.split() if w not in banned_set)       # (3) đếm từ hợp lệ
return count.most_common(1)[0][0]                                     # (4) từ nhiều nhất
```
- `(4)`: `most_common(1)` trả list một cặp `[(từ, số_lần)]` → `[0]` lấy cặp → `[0]` lấy từ.

In [10]:
def most_common_word(paragraph: str, banned: list[str]) -> str:
    banned_set = set(banned)
    text = "".join(c.lower() if c.isalpha() else " " for c in paragraph)
    count = Counter(w for w in text.split() if w not in banned_set)
    return count.most_common(1)[0][0]

print(most_common_word(p1, banned1), most_common_word(p2, banned2))
print(Counter(["x", "y", "x"]).most_common(1), "->", Counter(["x", "y", "x"]).most_common(1)[0][0])

ball a
[('x', 2)] -> x


## 7. TRACE từng bước

Ví dụ 1, sau chuẩn hóa ta có 13 từ. Bảng đếm (chỉ từ không bị cấm):

| bước | từ | bị cấm? | count sau |
|---|---|:---:|---|
| 1 | bob | | bob:1 |
| 2 | hit | ✔ | (bỏ qua) |
| 3 | a | | a:1 |
| 4 | ball | | ball:1 |
| 5 | the | | the:1 |
| 6 | hit | ✔ | (bỏ qua) |
| 7 | ball | | **ball:2** |
| 8–12 | flew, far, after, it, was | | mỗi từ :1 |
| 13 | hit | ✔ | (bỏ qua) |

In [11]:
def trace_most_common_word(paragraph, banned):
    banned_set = set(banned)
    text = "".join(c.lower() if c.isalpha() else " " for c in paragraph)
    print("sau chuẩn hóa:", repr(text))
    count = {}
    for step, w in enumerate(text.split(), start=1):
        if w in banned_set:
            print(f"  bước {step:>2}: {w!r:8} bị cấm -> bỏ qua")
            continue
        count[w] = count.get(w, 0) + 1
        print(f"  bước {step:>2}: {w!r:8} -> count[{w!r}] = {count[w]}")
    best = max(count, key=count.get)
    print("bảng đếm:", count)
    print("đáp án:", best, "với", count[best], "lần")
    return best

_ = trace_most_common_word(p1, banned1)

sau chuẩn hóa: 'bob hit a ball  the hit ball flew far after it was hit '
  bước  1: 'bob'    -> count['bob'] = 1
  bước  2: 'hit'    bị cấm -> bỏ qua
  bước  3: 'a'      -> count['a'] = 1
  bước  4: 'ball'   -> count['ball'] = 1
  bước  5: 'the'    -> count['the'] = 1
  bước  6: 'hit'    bị cấm -> bỏ qua
  bước  7: 'ball'   -> count['ball'] = 2
  bước  8: 'flew'   -> count['flew'] = 1
  bước  9: 'far'    -> count['far'] = 1
  bước 10: 'after'  -> count['after'] = 1
  bước 11: 'it'     -> count['it'] = 1
  bước 12: 'was'    -> count['was'] = 1
  bước 13: 'hit'    bị cấm -> bỏ qua
bảng đếm: {'bob': 1, 'a': 1, 'ball': 2, 'the': 1, 'flew': 1, 'far': 1, 'after': 1, 'it': 1, 'was': 1}
đáp án: ball với 2 lần


`continue` bỏ phần còn lại của vòng lặp hiện tại và **sang lượt kế tiếp** (khác `break` là thoát hẳn vòng lặp).

## 8. Code nộp LeetCode — lời giải chính

In [12]:
from collections import Counter

class Solution:
    def mostCommonWord(self, paragraph: str, banned: list[str]) -> str:
        banned_set = set(banned)
        text = "".join(c.lower() if c.isalpha() else " " for c in paragraph)
        count = Counter(w for w in text.split() if w not in banned_set)
        return count.most_common(1)[0][0]

sol = Solution()
assert sol.mostCommonWord(p1, banned1) == "ball"
assert sol.mostCommonWord(p2, banned2) == "a"
print("Solution OK")

Solution OK


## 9. Cách 3 — Biểu thức chính quy `re.findall`

Module chuẩn `re` (regular expression). `re.findall(r"[a-z]+", text)` trả về **mọi đoạn liên tiếp** gồm một hoặc nhiều chữ `a..z`:
- `[a-z]` = một ký tự trong khoảng `a..z`.
- `+` = lặp **một hoặc nhiều** lần.
- `r"..."` là **raw string**: dấu `\` không bị Python hiểu thành ký tự thoát.

Áp dụng sau khi `lower()` thì dấu câu và dấu cách tự bị bỏ.

In [13]:
import re
print(re.findall(r"[a-z]+", p1.lower()))
print(re.findall(r"\d+", "năm 2026 có 12 tháng"))      # \d = chữ số

class SolutionRegex:
    def mostCommonWord(self, paragraph: str, banned: list[str]) -> str:
        banned_set = set(banned)
        count = Counter(w for w in re.findall(r"[a-z]+", paragraph.lower()) if w not in banned_set)
        return max(count, key=count.get)

print(SolutionRegex().mostCommonWord(p1, banned1), SolutionRegex().mostCommonWord(p2, banned2))

['bob', 'hit', 'a', 'ball', 'the', 'hit', 'ball', 'flew', 'far', 'after', 'it', 'was', 'hit']
['2026', '12']
ball a


**Cẩn thận với `\w+`:** nhiều lời giải dùng `re.findall(r"\w+", ...)`. `\w` gồm chữ cái, chữ số **và dấu gạch dưới `_`**. Với constraints của bài (chỉ chữ cái + `!?',;.`) thì vẫn đúng, nhưng `[a-z]+` diễn đạt chính xác ý "chỉ chữ cái" hơn.

## 10. Cách 4 — `str.translate` thay dấu câu

`str.maketrans("!?',;.", "      ")` tạo bảng thay **từng ký tự** ở chuỗi thứ nhất bằng ký tự cùng vị trí ở chuỗi thứ hai (hai chuỗi phải cùng độ dài). Nhanh vì chạy trong C, nhưng **phụ thuộc** vào việc liệt kê đủ dấu câu theo đề.

In [14]:
PUNCT = "!?',;."
table = str.maketrans(PUNCT, " " * len(PUNCT))
print(p1.lower().translate(table).split())

def most_common_translate(paragraph, banned):
    banned_set = set(banned)
    words = paragraph.lower().translate(table).split()
    count = Counter(w for w in words if w not in banned_set)
    return count.most_common(1)[0][0]

print(most_common_translate(p1, banned1), most_common_translate(p2, banned2))

['bob', 'hit', 'a', 'ball', 'the', 'hit', 'ball', 'flew', 'far', 'after', 'it', 'was', 'hit']
ball a


## 11. Vì sao đúng?

1. **Tách từ đúng:** một "từ" theo đề là dãy chữ cái liên tiếp tối đa (dấu câu và dấu cách đều là ranh giới). Thay mọi ký tự không phải chữ cái bằng dấu cách rồi `split()` cho ra **chính xác** các dãy đó (`split()` bỏ chuỗi rỗng do nhiều dấu cách liền nhau).
2. **Không phân biệt hoa/thường:** `lower()` đưa mọi biến thể (`"BALL"`, `"Ball"`) về một khóa duy nhất `"ball"`; `banned` đã là chữ thường nên so sánh khớp.
3. **Đếm:** `Counter` cộng đúng số lần mỗi khóa; từ cấm không bao giờ được thêm.
4. **Chọn max:** đề bảo đảm đáp án duy nhất ⇒ `most_common(1)` / `max(..., key=...)` trả đúng từ đó (không lo hòa).

## 12. Độ phức tạp và so sánh

`P = len(paragraph)` (≤ 1000), `B` = tổng độ dài từ cấm (≤ 1000), `U` = số từ khác nhau.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Vòng ký tự + dict | `O(P + B)` | `O(P + B)` | Hiểu bản chất tokenize |
| 2. Chuẩn hóa + `split` + `Counter` | `O(P + B)` | `O(P + B)` | **Khuyến nghị** |
| 3. `re.findall` | `O(P + B)` | `O(P + B)` | Gọn, cần biết regex |
| 4. `translate` | `O(P + B)` | `O(P + B)` | Nhanh, phụ thuộc danh sách dấu câu |

`most_common(1)` tốn `O(U)` (dùng `max` nội bộ), `max(count, key=count.get)` cũng `O(U)`; `U ≤ P`.

In [15]:
import time
big_par = (p1 + " ") * 2000            # ~120 000 ký tự, vượt constraints để thấy xu hướng
for name, f in [("vòng ký tự", most_common_manual), ("chuẩn hóa+Counter", most_common_word),
                ("regex", lambda p, bn: SolutionRegex().mostCommonWord(p, bn)), ("translate", most_common_translate)]:
    t = time.perf_counter()
    r = f(big_par, banned1)
    print(f"{name:<18} -> {r}  ({(time.perf_counter() - t) * 1000:.1f} ms)")

vòng ký tự         -> ball  (9.8 ms)
chuẩn hóa+Counter  -> ball  (11.9 ms)
regex              -> ball  (5.1 ms)
translate          -> ball  (2.4 ms)


Tất cả đều tuyến tính; `translate` và `regex` nhanh nhất vì phần nặng chạy trong C. Với `paragraph ≤ 1000` ký tự, khác biệt không đáng kể — hãy chọn cách **dễ đúng** nhất.

## 13. Bộ test

1. Hai ví dụ đề.
2. Dấu câu **không có dấu cách sau** (`"a,b,b"`) — bẫy lớn nhất của bài!
3. Hoa/thường lẫn lộn.
4. Từ cấm xuất hiện nhiều nhất.
5. Đoạn văn chỉ một từ, có/không dấu câu cuối.
6. Dấu nháy đơn `'` (ví dụ `"Bob's"` → `bob` và `s`).
7. Đối chiếu 4 cách trên dữ liệu ngẫu nhiên (bảo đảm đáp án duy nhất).

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual!r}")

check("đề 1", most_common_word(p1, banned1), "ball")
check("đề 2", most_common_word(p2, banned2), "a")
check("dấu phẩy không cách", most_common_word("a, a, a, a, b,b,b,c, c", ["a"]), "b")
check("hoa thường", most_common_word("Bob. bOB! BOB? alice", []), "bob")
check("từ cấm nhiều nhất", most_common_word("x x x y", ["x"]), "y")
check("một từ", most_common_word("Hello", []), "hello")
check("nháy đơn", most_common_word("Bob's dog; Bob's cat. s s", ["s"]), "bob")

PASS đề 1: 'ball'
PASS đề 2: 'a'
PASS dấu phẩy không cách: 'b'
PASS hoa thường: 'bob'
PASS từ cấm nhiều nhất: 'y'
PASS một từ: 'hello'
PASS nháy đơn: 'bob'


In [17]:
import random
random.seed(819)
vocab = ["bob", "ball", "hit", "a", "the", "it", "far"]
seps = [" ", ", ", ",", ". ", "!", "? ", ";", "'", " ; "]
methods = [most_common_manual, most_common_word, most_common_translate,
           lambda p, bn: Solution().mostCommonWord(p, bn), lambda p, bn: SolutionRegex().mostCommonWord(p, bn)]
tested = 0
while tested < 300:
    ws = [random.choice(vocab) for _ in range(random.randint(1, 30))]
    bn = random.sample(vocab, random.randint(0, 3))
    cnt = Counter(w for w in ws if w not in bn)
    top = cnt.most_common(2)
    if not top or (len(top) == 2 and top[0][1] == top[1][1]):
        continue                                   # bỏ bộ không có đáp án duy nhất
    para = ""
    for w in ws:
        w2 = "".join(ch.upper() if random.random() < 0.3 else ch for ch in w)
        para += w2 + random.choice(seps)
    expected = top[0][0]
    for f in methods:
        assert f(para, bn) == expected, (f, para, bn)
    tested += 1
print("300 đoạn văn ngẫu nhiên (dấu câu, hoa/thường lộn xộn): 5 cách khớp nhau")

300 đoạn văn ngẫu nhiên (dấu câu, hoa/thường lộn xộn): 5 cách khớp nhau


## 14. Lỗi người mới hay gặp

### Lỗi 1: `split(" ")` rồi `strip` dấu câu ở hai đầu
```python
for w in paragraph.lower().split(" "):
    w = w.strip("!?',;.")      # SAI với "b,b,b": thành một "từ" 'b,b,b'
```

### Lỗi 2: Quên `lower()`
`"BALL"` và `"ball"` bị đếm như hai từ khác nhau.

### Lỗi 3: Dùng `banned` là list và kiểm tra `in` trong vòng lặp
Đúng kết quả nhưng `O(P · |banned|)`; dùng `set`.

### Lỗi 4: Lọc từ cấm **sau** khi lấy max
```python
best = Counter(words).most_common(1)[0][0]   # có thể chính là từ cấm!
```

### Lỗi 5: Quên từ cuối khi tự tokenize (đoạn văn kết thúc bằng chữ cái)

In [18]:
def wrong_split_strip(paragraph, banned):
    bs = set(banned)
    words = [w.strip("!?',;.") for w in paragraph.lower().split(" ")]
    return Counter(w for w in words if w and w not in bs).most_common(1)[0][0]

def wrong_filter_after(paragraph, banned):
    text = "".join(c.lower() if c.isalpha() else " " for c in paragraph)
    return Counter(text.split()).most_common(1)[0][0]

def wrong_tokenize_forget_last(paragraph):
    words, cur = [], []
    for ch in paragraph:
        if ch.isalpha():
            cur.append(ch.lower())
        elif cur:
            words.append("".join(cur)); cur = []
    return words                      # quên từ cuối

t = "a, a, a, a, b,b,b,c, c"
print("split+strip :", wrong_split_strip(t, ["a"]), "| đúng:", most_common_word(t, ["a"]))
print("lọc sau max :", wrong_filter_after(p1, banned1), "| đúng:", most_common_word(p1, banned1))
print("quên từ cuối:", wrong_tokenize_forget_last("Bob hit ball"), "| đúng:", tokenize_manual("Bob hit ball"))

split+strip : b,b,b,c | đúng: b
lọc sau max : hit | đúng: ball
quên từ cuối: ['bob', 'hit'] | đúng: ['bob', 'hit', 'ball']


## 15. Mẹo nhận dạng pattern

- "Từ xuất hiện nhiều nhất / tần suất" ⇒ **Counter** (hash map đếm).
- "Bỏ qua danh sách cấm / stop words" ⇒ **set** để tra `O(1)`.
- "Không phân biệt hoa thường, bỏ dấu câu" ⇒ **chuẩn hóa trước** (lower + thay ký tự không phải chữ bằng dấu cách), rồi mới xử lý.
- Tách từ an toàn: `re.findall(r"[a-z]+", s.lower())` hoặc "thay rồi `split()`" — tránh `split(" ")`.

### Biến thể

1. **Top K Frequent Words (LeetCode 692)**: k từ nhiều nhất, hòa thì theo thứ tự từ điển → sắp xếp theo khóa `(-count, word)`.
2. Trả về **tất cả** từ đồng hạng nhất (khi không bảo đảm duy nhất).
3. Đếm **số từ khác nhau** không bị cấm.
4. **Word Frequency** trên nhiều đoạn văn (gộp `Counter` bằng `+` hoặc `update`).

In [19]:
# Biến thể 1: top-k, hòa thì theo thứ tự từ điển
def top_k_words(paragraph, banned, k):
    bs = set(banned)
    cnt = Counter(w for w in re.findall(r"[a-z]+", paragraph.lower()) if w not in bs)
    return sorted(cnt, key=lambda w: (-cnt[w], w))[:k]

print(top_k_words("b a c b a d", [], 2))           # a và b cùng 2 lần -> theo từ điển: a, b
print(top_k_words(p1, banned1, 3))
print(sorted(["b", "a", "c"], key=lambda w: w), sorted([(2, "x"), (1, "y")]))

['a', 'b']
['ball', 'a', 'after']
['a', 'b', 'c'] [(1, 'y'), (2, 'x')]


`sorted(..., key=lambda w: (-cnt[w], w))`: Python so sánh **tuple** từng thành phần từ trái sang phải. `-cnt[w]` để số lần **lớn** đứng trước; hòa thì so `w` theo từ điển.

## 16. Bài tập tự luyện (TODO — đáp án ở cuối)

**Bài 1 (đoán output):** Không chạy code, cho biết danh sách từ (sau chuẩn hóa) của `"Hi!hi, HI;ho.Ho"` và đáp án `mostCommonWord` với `banned = []`. Gán `answer_ex1 = (list_từ, đáp_án)`.

**Bài 2 (điền chỗ trống):** Hoàn thiện `tokenize_exercise(paragraph)` bằng "thay ký tự không phải chữ cái rồi `split()`".

**Bài 3 (viết lại):** `most_common_word_dict(paragraph, banned)` không dùng `Counter`, chỉ dùng `dict` và `get`.

**Bài 4 (biến thể):** `all_top_words(paragraph, banned)` trả về list **đã sắp xếp** các từ có tần suất cao nhất (có thể nhiều từ).

**Bài 5 (mini challenge):** `top_k_frequent(words, k)` (LeetCode 692) — cho list từ đã chuẩn hóa, trả `k` từ nhiều nhất; hòa thì theo thứ tự từ điển.

In [20]:
answer_ex1 = None   # TODO: (list, str)

def tokenize_exercise(paragraph: str) -> list[str]:
    # TODO
    pass

def most_common_word_dict(paragraph: str, banned: list[str]) -> str:
    # TODO
    pass

# assert tokenize_exercise("a.B,c") == ["a", "b", "c"]
# assert most_common_word_dict(p1, banned1) == "ball"

In [21]:
def all_top_words(paragraph: str, banned: list[str]) -> list[str]:
    # TODO
    pass

def top_k_frequent(words: list[str], k: int) -> list[str]:
    # TODO
    pass

# assert all_top_words("a b a b c", []) == ["a", "b"]
# assert top_k_frequent(["i","love","leetcode","i","love","coding"], 2) == ["i", "love"]

### Gợi ý 3 mức

**Bài 1** — (1) Mọi dấu `! , ; .` đều là ranh giới, kể cả khi không có dấu cách. (2) Đổi hết thành chữ thường. (3) Các từ: `hi` ba lần, `ho` hai lần.

**Bài 2** — (1) Duyệt từng ký tự. (2) Chữ cái → `lower()`, còn lại → `" "`. (3) `"".join(...)` rồi `.split()`.

**Bài 3** — (1) `count = {}`. (2) `count[w] = count.get(w, 0) + 1`. (3) `max(count, key=count.get)`.

**Bài 4** — (1) Đếm như thường. (2) `m = max(count.values())`. (3) `sorted(w for w, c in count.items() if c == m)`.

**Bài 5** — (1) `Counter(words)`. (2) Sắp xếp theo `(-count, word)`. (3) Lấy `[:k]`.

---

## 17. Đáp án tham khảo

In [22]:
answer_ex1_solution = (["hi", "hi", "hi", "ho", "ho"], "hi")

def tokenize_exercise_solution(paragraph):
    return "".join(c.lower() if c.isalpha() else " " for c in paragraph).split()

def most_common_word_dict_solution(paragraph, banned):
    bs = set(banned)
    count = {}
    for w in tokenize_exercise_solution(paragraph):
        if w not in bs:
            count[w] = count.get(w, 0) + 1
    return max(count, key=count.get)

def all_top_words_solution(paragraph, banned):
    bs = set(banned)
    count = Counter(w for w in tokenize_exercise_solution(paragraph) if w not in bs)
    m = max(count.values())
    return sorted(w for w, c in count.items() if c == m)

def top_k_frequent_solution(words, k):
    cnt = Counter(words)
    return sorted(cnt, key=lambda w: (-cnt[w], w))[:k]

ex = "Hi!hi, HI;ho.Ho"
assert (tokenize_exercise_solution(ex), most_common_word(ex, [])) == answer_ex1_solution
assert tokenize_exercise_solution("a.B,c") == ["a", "b", "c"]
assert most_common_word_dict_solution(p1, banned1) == "ball"
assert most_common_word_dict_solution(p2, banned2) == "a"
assert all_top_words_solution("a b a b c", []) == ["a", "b"]
assert all_top_words_solution(p1, banned1) == ["ball"]
assert top_k_frequent_solution(["i", "love", "leetcode", "i", "love", "coding"], 2) == ["i", "love"]
assert top_k_frequent_solution(["the", "day", "is", "sunny", "the", "the", "the",
                                "sunny", "is", "is"], 4) == ["the", "is", "sunny", "day"]
print("Đáp án PASS toàn bộ test!")

Đáp án PASS toàn bộ test!


## 18. Template nhớ nhanh và checklist submit

```python
from collections import Counter

class Solution:
    def mostCommonWord(self, paragraph: str, banned: list[str]) -> str:
        banned_set = set(banned)
        text = "".join(c.lower() if c.isalpha() else " " for c in paragraph)
        return Counter(w for w in text.split() if w not in banned_set).most_common(1)[0][0]
```

Công thức nhớ: **chuẩn hóa → tách → lọc (set) → đếm (Counter) → max**.

**Checklist trước khi submit:**
- [ ] `lower()` trước khi đếm?
- [ ] Dấu câu **không có dấu cách theo sau** (`"b,b,b"`) vẫn tách đúng?
- [ ] Dùng `split()` không tham số (bỏ chuỗi rỗng)?
- [ ] Lọc từ cấm **trước** khi lấy max; `banned` chuyển thành `set`?
- [ ] Trả về **chữ thường**?